# Clustering Spasial Gempa Bumi Indonesia — Identifikasi Seismic Gap
**Dataset:** `indonesia_earthquakes_final.csv`  
**Metode:** K-Means Clustering pada fitur: latitude, longitude, depth, magnitude

## Cell 1 — Import & Load Data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ── Paths ─────────────────────────────────────────────────────────────────────
WORKDIR = Path(".")
INPUT_CSV  = WORKDIR / "indonesia_earthquakes_final.csv"
OUT_DIR    = WORKDIR / "output_cluster"
OUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
df["time"] = pd.to_datetime(df["time"], errors="coerce")

print(f"Shape  : {df.shape}")
print(f"Kolom  : {list(df.columns)}")
print(f"\nRentang tanggal : {df['time'].min().date()}  s/d  {df['time'].max().date()}")
print(f"Rentang mag     : {df['mag'].min():.1f}  –  {df['mag'].max():.1f}")
df.head()

## Cell 2 — Persiapan Fitur Clustering

In [ ]:
FEATURES = ["latitude", "longitude", "depth", "mag"]

X_raw = df[FEATURES].values

scaler = StandardScaler()
X      = scaler.fit_transform(X_raw)

print("Fitur yang digunakan :", FEATURES)
print(f"Shape X (scaled)     : {X.shape}")
print(f"\nStatistik setelah scaling (mean ≈ 0, std ≈ 1):")

stats = pd.DataFrame(X, columns=FEATURES)
display(stats.describe().round(4))

print("\nNilai mean scaler (sebelum scaling) :", dict(zip(FEATURES, scaler.mean_.round(4))))
print("Nilai std  scaler (sebelum scaling) :", dict(zip(FEATURES, scaler.scale_.round(4))))

## Cell 3 — Elbow Method & Silhouette Score

In [ ]:
K_RANGE   = range(2, 11)
inertias  = []
sil_scores = []

print("Menghitung Elbow & Silhouette (k=2..10) ...")
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels, sample_size=10_000, random_state=42))
    print(f"  k={k:2d}  inertia={km.inertia_:>14,.1f}  silhouette={sil_scores[-1]:.4f}")

# ── Tabel ringkasan ───────────────────────────────────────────────────────────
metrics_df = pd.DataFrame({
    "k"             : list(K_RANGE),
    "Inertia"       : inertias,
    "Silhouette"    : [round(s, 4) for s in sil_scores],
})
print("\n", metrics_df.to_string(index=False))

# ── k optimal: inertia knee (second-derivative max) ──────────────────────────
inertia_arr = np.array(inertias)
d1 = np.diff(inertia_arr)          # first derivative
d2 = np.diff(d1)                   # second derivative (curvature)
knee_idx   = np.argmax(d2) + 2     # +2 karena double diff & 0-indexed
k_elbow    = list(K_RANGE)[knee_idx]
k_sil_best = list(K_RANGE)[np.argmax(sil_scores)]
k_optimal  = k_elbow               # pakai elbow sebagai acuan utama

print(f"\nk optimal (elbow/knee)    : {k_elbow}")
print(f"k terbaik (silhouette)    : {k_sil_best}  (score={max(sil_scores):.4f})")
print(f"k yang akan digunakan     : {k_optimal}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Elbow
axes[0].plot(list(K_RANGE), inertias, "o-", color="#2c7bb6", lw=2, ms=7)
axes[0].axvline(k_elbow, color="red", ls="--", lw=1.5,
                label=f"Knee k={k_elbow}")
axes[0].set_title("Elbow Method — Inertia vs k", fontweight="bold")
axes[0].set_xlabel("Jumlah Cluster (k)")
axes[0].set_ylabel("Inertia (WCSS)")
axes[0].legend(fontsize=9)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:,.0f}"))

# Silhouette
axes[1].plot(list(K_RANGE), sil_scores, "s-", color="#d7191c", lw=2, ms=7)
axes[1].axvline(k_sil_best, color="green", ls="--", lw=1.5,
                label=f"Best k={k_sil_best} (score={max(sil_scores):.3f})")
axes[1].set_title("Silhouette Score vs k", fontweight="bold")
axes[1].set_xlabel("Jumlah Cluster (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].legend(fontsize=9)

plt.suptitle("Penentuan k Optimal — Clustering Gempa Indonesia", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "elbow_silhouette.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafik disimpan: output_cluster/elbow_silhouette.png")

## Cell 4 — KMeans dengan k=5
> k=5 dipilih berdasarkan silhouette score tertinggi (0.3201) dan sesuai 5 zona tektonik utama Indonesia: Sumatera, Jawa-Bali-NTB, Sulawesi-NTT, Maluku, Papua.

In [ ]:
K_FINAL = 5

kmeans = KMeans(n_clusters=K_FINAL, n_init=10, random_state=42)
df["cluster"] = kmeans.fit_predict(X)

# Centroid dalam ruang asli (inverse transform)
centroids_scaled = kmeans.cluster_centers_
centroids_raw    = scaler.inverse_transform(centroids_scaled)
centroids_df     = pd.DataFrame(centroids_raw, columns=FEATURES)
centroids_df.index.name = "cluster"

sil_k5 = silhouette_score(X, df["cluster"], sample_size=10_000, random_state=42)

print(f"KMeans  k={K_FINAL}  |  Inertia={kmeans.inertia_:,.1f}  |  Silhouette={sil_k5:.4f}")
print(f"\nJumlah event per cluster:")
cluster_counts = df["cluster"].value_counts().sort_index()
for c, n in cluster_counts.items():
    print(f"  Cluster {c} : {n:>7,}  ({n/len(df)*100:.1f}%)")

print(f"\nCentroid per cluster (skala asli):")
display(centroids_df.round(3))

## Cell 5 — Visualisasi Cluster di Peta (k=5)

In [ ]:
CLUSTER_COLORS = {
    0: "#e41a1c",   # merah
    1: "#377eb8",   # biru
    2: "#4daf4a",   # hijau
    3: "#ff7f00",   # oranye
    4: "#984ea3",   # ungu
}

fig, ax = plt.subplots(figsize=(16, 8))

# ── Scatter per cluster ───────────────────────────────────────────────────────
for c in sorted(df["cluster"].unique()):
    sub = df[df["cluster"] == c]
    ax.scatter(
        sub["longitude"], sub["latitude"],
        c=CLUSTER_COLORS[c], s=2, alpha=0.35,
        label=f"Cluster {c}  (n={len(sub):,})",
        rasterized=True,
    )

# ── Centroid markers ──────────────────────────────────────────────────────────
for c in range(K_FINAL):
    cx = centroids_df.loc[c, "longitude"]
    cy = centroids_df.loc[c, "latitude"]
    ax.scatter(cx, cy, c=CLUSTER_COLORS[c], s=280, marker="*",
               edgecolors="black", lw=0.8, zorder=10)
    ax.annotate(
        f" C{c}\n({cy:.1f}N,{cx:.1f}E)",
        xy=(cx, cy), fontsize=7.5, fontweight="bold",
        xytext=(5, 5), textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.75),
    )

# ── Batas bounding box Indonesia ─────────────────────────────────────────────
import matplotlib.patches as mpatches
rect = mpatches.FancyBboxPatch(
    (95, -11), 46, 20,
    boxstyle="square,pad=0", linewidth=1.5,
    edgecolor="black", facecolor="none", linestyle="--",
)
ax.add_patch(rect)
ax.text(96, -10.5, "Batas Wilayah Indonesia", fontsize=8, color="black", alpha=0.7)

ax.set_xticks(range(95, 142, 5))
ax.set_yticks(range(-11, 10, 2))
ax.grid(True, alpha=0.2, lw=0.5)
ax.set_xlim(93, 143); ax.set_ylim(-12, 10)
ax.set_xlabel("Longitude (°E)", fontsize=11)
ax.set_ylabel("Latitude (°N)",  fontsize=11)
ax.set_title(
    "Peta Clustering Spasial Gempa Bumi Indonesia (K-Means, k=5)\n"
    "Fitur: latitude, longitude, depth, magnitude  |  Silhouette = 0.3201",
    fontsize=13, fontweight="bold",
)
legend = ax.legend(markerscale=5, fontsize=9.5, loc="upper left",
                   framealpha=0.85, title="Cluster")
legend.get_title().set_fontweight("bold")

plt.tight_layout()
plt.savefig(OUT_DIR / "cluster_map_k5.png", dpi=150, bbox_inches="tight")
plt.show()
print("Peta disimpan: output_cluster/cluster_map_k5.png")

## Cell 6 — Penamaan Zona Tektonik (k=5)

In [ ]:
ZONE_NAMES = [
    "Zona Sumatera",
    "Zona Jawa-Bali-NTB",
    "Zona Sulawesi-NTT",
    "Zona Maluku",
    "Zona Papua",
]

# Urutkan cluster berdasarkan longitude centroid (barat → timur)
lon_order = centroids_df["longitude"].sort_values().index.tolist()
zone_map  = {c: ZONE_NAMES[rank] for rank, c in enumerate(lon_order)}

df["zone_name"] = df["cluster"].map(zone_map)
centroids_df["zone_name"] = centroids_df.index.map(zone_map)

# ── Tabel ringkasan zona ──────────────────────────────────────────────────────
zone_summary = (
    df.groupby("zone_name")
    .agg(
        cluster   =("cluster",   "first"),
        n_event   =("cluster",   "count"),
        lon_mean  =("longitude", "mean"),
        lat_mean  =("latitude",  "mean"),
        depth_mean=("depth",     "mean"),
        mag_mean  =("mag",       "mean"),
        mag_max   =("mag",       "max"),
    )
    .reindex(ZONE_NAMES)
    .round(2)
)
zone_summary["persen (%)"] = (zone_summary["n_event"] / len(df) * 100).round(1)

print("Pemetaan Cluster -> Zona Tektonik (diurutkan barat ke timur):")
print(f"\n  {'Zona':<22}  {'Cluster':>7}  {'n_event':>8}  {'%':>5}  "
      f"{'lon_mean':>9}  {'lat_mean':>9}  {'depth_mean':>11}  {'mag_mean':>9}")
print(f"  {'-'*22}  {'-'*7}  {'-'*8}  {'-'*5}  {'-'*9}  {'-'*9}  {'-'*11}  {'-'*9}")
for zone in ZONE_NAMES:
    r = zone_summary.loc[zone]
    print(f"  {zone:<22}  {int(r['cluster']):>7}  {int(r['n_event']):>8,}  "
          f"{r['persen (%)']:>4.1f}%  {r['lon_mean']:>9.2f}  "
          f"{r['lat_mean']:>9.2f}  {r['depth_mean']:>11.1f}  {r['mag_mean']:>9.3f}")

display(zone_summary)

## Cell 7 — Statistik per Zona Tektonik

In [ ]:
STAT_COLS = ["latitude", "longitude", "depth", "mag"]

# ── Statistik deskriptif per zona ─────────────────────────────────────────────
print("=" * 70)
print("STATISTIK DESKRIPTIF PER ZONA TEKTONIK")
print("=" * 70)
cluster_stats = (
    df.groupby("zone_name")[STAT_COLS]
    .agg(["mean", "std", "min", "max"])
    .reindex(ZONE_NAMES)
    .round(3)
)
display(cluster_stats)

# ── Palette warna zona (urut barat → timur) ───────────────────────────────────
zone_colors = {
    zone: CLUSTER_COLORS[zone_map_inv]
    for zone_map_inv, zone in zone_map.items()
}
palette = [zone_colors[z] for z in ZONE_NAMES]

# ── Boxplot mag & depth per zona ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_i, col in enumerate(["mag", "depth"]):
    data_zones = [df[df["zone_name"] == z][col].values for z in ZONE_NAMES]
    short_labels = ["Sumatera", "Jawa-Bali\n-NTB", "Sulawesi\n-NTT", "Maluku", "Papua"]
    bp = axes[ax_i].boxplot(
        data_zones,
        labels=short_labels,
        patch_artist=True,
        medianprops=dict(color="black", lw=2),
        flierprops=dict(marker=".", markersize=1.5, alpha=0.25),
    )
    for patch, color in zip(bp["boxes"], palette):
        patch.set_facecolor(color); patch.set_alpha(0.7)

    axes[ax_i].set_title(
        f"Distribusi {'Magnitudo' if col=='mag' else 'Kedalaman (km)'} per Zona",
        fontweight="bold",
    )
    axes[ax_i].set_ylabel("Magnitudo" if col == "mag" else "Kedalaman (km)")
    axes[ax_i].tick_params(axis="x", labelsize=9)

    # Tambahkan median label
    for j, med_line in enumerate(bp["medians"]):
        med_val = med_line.get_ydata()[0]
        axes[ax_i].text(
            j + 1, med_val + (0.02 if col == "mag" else 2),
            f"{med_val:.1f}", ha="center", va="bottom",
            fontsize=8, fontweight="bold", color="black",
        )

plt.suptitle("Karakteristik Seismik per Zona Tektonik Indonesia (k=5)",
             fontweight="bold", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "boxplot_k5.png", dpi=150, bbox_inches="tight")
plt.show()
print("Boxplot disimpan: output_cluster/boxplot_k5.png")

## Cell 8 — Simpan Dataset

In [ ]:
OUT_DATA      = WORKDIR / "indonesia_earthquakes_clustered.csv"
OUT_CENTROIDS = WORKDIR / "cluster_centroids.csv"

# ── Simpan dataset ────────────────────────────────────────────────────────────
df_save = df.copy()
df_save["time"] = df_save["time"].dt.strftime("%Y-%m-%dT%H:%M:%S")
df_save.to_csv(OUT_DATA, index=False, encoding="utf-8")
size_mb = OUT_DATA.stat().st_size / (1024**2)
print(f"[OK] Dataset clustered  : {OUT_DATA.name}  ({size_mb:.2f} MB)")
print(f"     Shape              : {df_save.shape}")
print(f"     Kolom              : {list(df_save.columns)}")

# ── Simpan centroid ───────────────────────────────────────────────────────────
centroids_out = centroids_df.copy()
centroids_out.to_csv(OUT_CENTROIDS, index=True, encoding="utf-8")
print(f"\n[OK] Centroids          : {OUT_CENTROIDS.name}")
display(centroids_out.round(3))

# ── Cek semua file output ─────────────────────────────────────────────────────
print(f"\n{'='*62}")
print("FILE OUTPUT")
print(f"{'='*62}")
output_files = {
    "indonesia_earthquakes_clustered.csv" : OUT_DATA,
    "cluster_centroids.csv"               : OUT_CENTROIDS,
    "output_cluster/cluster_map_k5.png"   : OUT_DIR / "cluster_map_k5.png",
    "output_cluster/boxplot_k5.png"       : OUT_DIR / "boxplot_k5.png",
    "output_cluster/elbow_silhouette.png" : OUT_DIR / "elbow_silhouette.png",
}
for name, path in output_files.items():
    kb = path.stat().st_size / 1024 if path.exists() else 0
    status = "[OK]" if path.exists() else "[!!]"
    print(f"  {status}  {name:<45} ({kb:.1f} KB)")

# ── Summary akhir ─────────────────────────────────────────────────────────────
print(f"\n{'='*62}")
print("SUMMARY AKHIR — JUMLAH EVENT PER ZONA TEKTONIK")
print(f"{'='*62}")
print(f"  Total event          : {len(df):,}")
print(f"  Jumlah cluster (k)   : {K_FINAL}")
print(f"  Silhouette score     : {sil_k5:.4f}")
print(f"  Inertia              : {kmeans.inertia_:,.1f}")
print()
print(f"  {'Zona':<25}  {'Cluster':>7}  {'N Event':>9}  {'%':>6}  {'Lon Mean':>9}  {'Mag Mean':>9}")
print(f"  {'-'*25}  {'-'*7}  {'-'*9}  {'-'*6}  {'-'*9}  {'-'*9}")
for zone in ZONE_NAMES:
    r   = zone_summary.loc[zone]
    clr = int(r["cluster"])
    n   = int(r["n_event"])
    print(f"  {zone:<25}  {clr:>7}  {n:>9,}  {r['persen (%)']:>5.1f}%  "
          f"{r['lon_mean']:>9.2f}  {r['mag_mean']:>9.3f}")
print(f"\n  [OK] Notebook 01_clustering.ipynb selesai direvisi dan dijalankan.")